# Zero-shot classification with CLIP

**Computer Vision in Archaeology Training School, Brno 2026**  
Monday morning, *Introduction to computer vision*.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arubrno/atrium-school-ml-lessons/blob/main/1-monday-intro/clip_zero_shot.ipynb)

Runs unchanged in Google Colab, on the school JupyterHub, and in any local
Jupyter or IDE. The first cell works out which of those you are in.

CLIP was trained on image–caption pairs scraped from the web. It never saw an
archaeological training set, and it was never told what an antoninianus is.

You hand it an image and a **list of candidate descriptions in plain English**;
it tells you which description best fits. No training, no annotation, no labels.

This notebook does three things:

1. runs CLIP on a few archaeological photographs;
2. shows that **rewording a label changes the answer** — the vocabulary is now a parameter;
3. shows that if you offer it **only wrong options, it picks one anyway, confidently**.

The third point is the one that matters.

## 1. Setup

Run the next cell and read what it prints. It finds the repository (cloning it
first if you are in Colab), installs only the packages that are missing, and
points the model cache somewhere sensible for wherever you are running.

Nothing below this cell knows or cares which platform you chose.

<details>
<summary><b>Colab</b> — what to expect</summary>

The clone takes a few seconds; PyTorch is already installed there. Colab wipes
its disk when the runtime is recycled, so the 600 MB of CLIP weights come down
again in a new session. To avoid that, run `setup("torch", "transformers",
drive=True)` — it asks permission to mount your Google Drive and keeps the
caches in `MyDrive/atrium-school`.

</details>

<details>
<summary><b>School JupyterHub</b> — nothing to set up</summary>

You get a URL on the first morning that drops you straight into JupyterLab, with
the server running and this repository already in place. Open the day's folder in
the file browser and double-click the notebook. The setup cell finds everything where
it expects it and installs nothing.

Your home directory persists between sessions, so your edits survive. If a
notebook stops responding, *Kernel → Restart Kernel* and re-run from the top.

</details>

<details>
<summary><b>Your own machine</b> — the one-time install</summary>

Follow the [setup guide](https://arup-cas.github.io/atrium-school-ml/setup.html):
clone the repository, make a virtual environment, `pip install -r
requirements.txt` with the PyTorch CPU index. Then start Jupyter anywhere inside
the clone. The setup cell will find the repository around it and install nothing.

</details>

In [ ]:
# --- ATRIUM bootstrap: identical in every notebook of this school ------------
# Colab starts with none of this repository, so this cell has to be able to
# fetch it before it can import anything of ours. Everything else lives in
# atrium_bootstrap.py at the repository root.
import pathlib, subprocess, sys

REPO = "https://github.com/arubrno/atrium-school-ml-lessons.git"

root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "atrium_bootstrap.py").exists()), None)
if root is None:
    root = pathlib.Path("atrium-school-ml-lessons").resolve()
    if not root.exists():
        print("fetching the course repository ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(root)], check=True)

sys.path.insert(0, str(root))
from atrium_bootstrap import setup

# drive=True on Colab keeps the model weights in your Google Drive between sessions.
setup("torch", "transformers")

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import requests, io
import matplotlib.pyplot as plt

MODEL_ID = "openai/clip-vit-base-patch32"

model = CLIPModel.from_pretrained(MODEL_ID).eval()
processor = CLIPProcessor.from_pretrained(MODEL_ID)

print(f"Loaded {MODEL_ID} on {'GPU' if torch.cuda.is_available() else 'CPU'} "
      f"({sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters)")


> **Before the session.** Run cells 1–3 once, on whatever you plan to use on the
> day. The weights download once and are then read from the cache — instantly on
> the hub or your laptop, and on Colab too as long as the runtime stays alive. If
> you skip this, expect a few minutes of silence while 600 MB comes down.

## 2. The images

Every dataset this week is reached the same way — `get_dataset(name)`. Where it
actually lives (in the repository, on the hub, or downloaded and cached) is
described once in `datasets.yml` at the repository root, and no notebook
needs to know.

To add your own photographs to this demo, drop them in `1-monday-intro/demo-images/`.

In [ ]:
from atrium_data import get_dataset   # the setup cell put the repo root on sys.path

DEMO = get_dataset("demo")
IMAGES = {p.stem: p for p in sorted(DEMO.glob("*.jpg"))}
print(f"{len(IMAGES)} images:", ", ".join(IMAGES))


def load(source):
    """Load an image from a local path or a URL."""
    source = str(source)
    if source.startswith(("http://", "https://")):
        r = requests.get(source, timeout=30)
        r.raise_for_status()
        return Image.open(io.BytesIO(r.content)).convert("RGB")
    return Image.open(source).convert("RGB")

## 3. The whole method, in one function

CLIP embeds the image and each caption into the same vector space, then scores
how close they are. `softmax` turns those scores into numbers that sum to 1.

Note what that means: **the probabilities are over the labels you supplied.**
They are not a confidence that the object is really there.

In [ ]:
def zero_shot(image, labels):
    """Return [(label, probability), ...] sorted best first."""
    inputs = processor(text=labels, images=image, return_tensors="pt", padding=True)
    with torch.no_grad():
        probs = model(**inputs).logits_per_image.softmax(dim=1)[0]
    return sorted(zip(labels, probs.tolist()), key=lambda kv: -kv[1])


BLUE, MUTE = "#2c7be5", "#6b7280"

def show(image, labels, title=None):
    """Photograph on the left, label probabilities on the right."""
    result = zero_shot(image, labels)
    fig, (ax_img, ax_bar) = plt.subplots(
        1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1, 1.7]})

    ax_img.imshow(image); ax_img.axis("off")

    names = [l for l, _ in result][::-1]
    vals  = [p for _, p in result][::-1]
    ax_bar.barh(names, vals, color=BLUE, height=0.62)
    for y, v in enumerate(vals):
        ax_bar.text(v + 0.015, y, f"{v:.0%}", va="center", color=MUTE, fontsize=10)
    ax_bar.set_xlim(0, 1.18)
    ax_bar.set_xticks([])
    for side in ("top", "right", "bottom", "left"):
        ax_bar.spines[side].set_visible(False)
    if title:
        ax_bar.set_title(title, loc="left", fontsize=11, fontweight="bold")

    plt.tight_layout(); plt.show()
    return result


## 4. A sensible set of labels

Four plausible options, one of them right.

In [ ]:
labels = [
    "a photo of a potsherd",
    "a photo of a coin",
    "a photo of a stone tool",
    "a photo of a scale bar",
]

img = load(IMAGES["sherd_01"])
show(img, labels, "Sensible options");


## 5. The vocabulary is a parameter

Same photograph, same model, three ways of saying the same thing.
Watch the numbers move.

There is no "correct" wording. Whatever you choose becomes part of your method,
and it has to go in the paper.

In [ ]:
phrasings = {
    "plain":       ["potsherd", "coin", "stone tool", "scale bar"],
    "a photo of":  ["a photo of a potsherd", "a photo of a coin",
                    "a photo of a stone tool", "a photo of a scale bar"],
    "specialist":  ["a fragment of wheel-thrown pottery", "an ancient bronze coin",
                    "a knapped flint tool", "a photographic scale bar"],
}

for name, ls in phrasings.items():
    show(img, ls, f"Phrasing: {name}")


## 6. Only wrong options

Now take the same sherd and offer the model nothing that fits.

It will not say *"none of these"*. It has no way to. It will pick the least-bad
option and report a probability that looks exactly as convincing as a correct one.

In [ ]:
wrong_only = [
    "a photo of a coin",
    "a photo of a stone tool",
    "a photo of a bicycle",
]

result = show(img, wrong_only, "Only wrong options")
top, p = result[0]
print(f"\nThe model's answer: {top!r} at {p:.0%} confidence.")
print("There is no potsherd option. There is no potsherd in the answer.")


## 7. What to take from this

Zero-shot is a superb way to **explore** a collection you have not labelled:
triage a folder, find the frames that contain a scale bar, get a first pass at
*"which of these 8 000 photographs are worth my afternoon"*.

It is a poor way to **report a number**, because:

- the probabilities are over *your* label list, not over reality;
- rewording a label changes the answer;
- there is no "none of the above", so absence is never reported;
- the training data is the open internet, with all of its biases about what
  archaeological material looks like and which parts of the world it comes from.

By Thursday you will be measuring exactly these failures with precision, recall
and mAP instead of eyeballing them.

## 8. Try it on your own image

Point at a photograph — a path, or a URL — and give it your own list of candidate
descriptions.

- **Colab**: the folder icon in the left sidebar has an upload button; a file you
  upload lands in `/content`, so `MY_IMAGE = "my_photograph.jpg"` finds it.
- **Jupyter / local**: put the file next to this notebook, or give a full path.
- **Anywhere**: a public image URL works too, with no upload at all.

Remember what section 6 showed: whatever you leave off the list cannot be the answer.

In [ ]:
MY_IMAGE = "my_photograph.jpg"   # a local path, or a URL

MY_LABELS = [
    "a photo of a potsherd",
    "a photo of a coin",
    "a photo of a stone tool",
]

show(load(MY_IMAGE), MY_LABELS, MY_IMAGE)